In [17]:
import pandas as pd

# Composite Scoring

## WSI01

In [ ]:
wsi01 = pd.read_excel("wsi01_combined_input.xlsx")

In [ ]:
lung_res = pd.read_excel("pre_lin_filt_OS_results.xlsx", sheet_name="RankedCellData")

# + 1 to account for header row
lung_res = lung_res.head(int(len(lung_res) * 0.01) + 1).reset_index(drop=True)

configs = lung_res[1:]["Features"]

In [ ]:
wsi01 = wsi01[wsi01["Features"].isin(configs)].reset_index(drop=True)

wsi01 = wsi01.merge(lung_res["Features"][1:], on="Features", how="outer").fillna(0)

In [59]:
param_dict = {}
for config in configs:

    mean = lung_res[lung_res["Features"] == config].iloc[:, 1:].mean(axis=1).values[0]
    std = lung_res[lung_res["Features"] == config].iloc[:, 1:].std(axis=1).values[0]

    params = {config: {"mean": mean, "std": std}}

    param_dict.update(params)

In [60]:
cols = wsi01.columns[1:]

for config in wsi01["Features"]:

    wsi01.loc[wsi01["Features"] == config, cols] = (wsi01.loc[wsi01["Features"] == config, cols] - param_dict[config]["mean"]) / param_dict[config]["std"]

In [61]:
wsi01

,Features,sampleid_1,sampleid_2,sampleid_3,sampleid_4,sampleid_5,sampleid_6,sampleid_7,sampleid_8,sampleid_9,...,sampleid_68,sampleid_69,sampleid_71,sampleid_72,sampleid_73,sampleid_74,sampleid_76,sampleid_77,sampleid_78,sampleid_79
0,0_0_0_0_1_total_25,197.917777,55.316435,20.563264,18.118005,62.228858,55.527129,213.462580,36.227209,6.712632,...,2.490325,40.272443,25.972471,0.575079,42.481511,93.684091,146.127372,110.432188,99.388511,70.198283
1,0_0_0_1_0_tumor_25,-1.158422,-1.158422,-1.156787,-1.158422,-1.106570,-1.157613,-1.158422,-1.158422,-1.152535,...,-1.158422,-1.151418,-1.158422,-1.158422,-1.158422,-1.133358,-1.114940,-1.004454,-1.157598,-1.154028
2,0_0_10_1_0_total_25,-0.834239,-0.832240,-0.834239,-0.834239,-0.834239,-0.833852,-0.834239,-0.834239,-0.834239,...,-0.834239,-0.834239,-0.834239,-0.834239,-0.834239,-0.834239,-0.834239,-0.834239,-0.834239,-0.834239
3,0_0_10_3_0_total_25,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,...,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594,-0.453594
4,0_0_10_3_0_tumor_25,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,...,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701,-0.679701
5,0_0_11_4_0_total_25,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,...,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089,-0.380089
6,0_0_14_4_0_total_25,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,...,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443,-0.380443
7,0_0_1_0_0_tumor_10,-1.672740,-1.683912,-1.696314,-1.733947,-1.614127,-1.723680,-1.737563,-1.738623,-1.659271,...,-1.736861,-1.717693,-1.715554,-1.723771,-1.697780,-1.733483,-1.603815,-1.635194,-1.730579,-1.654846
8,0_0_1_0_1_total_10,38.902136,42.325214,32.059330,18.244175,41.098272,12.859539,7.884266,1.404632,21.460345,...,1.635739,21.779927,26.903880,2.105622,11.244092,3.976441,14.062556,75.485336,15.812502,96.658066
9,0_0_1_1_0_total_175,-2.116494,-2.081079,-2.091376,-2.120157,-2.053260,-2.117983,-2.123690,-2.123690,-2.122133,...,-2.123690,-2.123094,-2.123690,-2.123690,-2.123690,-2.123690,-2.108864,-2.101974,-2.119624,-2.118401


In [62]:
rank_df = pd.read_excel("../Data/output/Axis/Pre/OS/pre_lin_filt_OS_results.xlsx", sheet_name="RankedFeatures")

rank_df = rank_df.head(int(len(rank_df) * 0.01)).reset_index(drop=True)

rank_cols = ["MI", "RankFeaturesROC", "RankFeaturesWil", "Relieff", "TreeBagger", "Fitctree", "ensembleBag", "ensembleGentleBoost", "ensembleRUSBoost", "Sequential_knn", "Sequential_tree", "Sequential_ensbl", "Sequential_svm", "Sequential_quadrLDA"]

rank_df.loc[:, "mean_performance"] = rank_df[rank_cols].mean(axis=1)


data = lung_res.T.reset_index()

data.columns = data.iloc[0]

data = data.drop(index=0)
data = data.reset_index(drop=True)

# Assess directionality by median density
directions = []
for feature in rank_df["OS"]:

    pos = data[data["OS"] == "Long"][feature].median()
    neg = data[data["OS"] == "Short"][feature].median()

    if pos < neg:
        directions.append(-1)
    else:
        directions.append(1)
    
rank_df.loc[:, "direction"] = directions


rank_df.loc[:, "weight"] = rank_df["mean_performance"] * rank_df["direction"]

rank_df = rank_df[["OS", "weight"]]

rank_df = rank_df.set_index("OS")["weight"]

In [66]:
wsi01_trans = wsi01.T.reset_index()

wsi01_trans.columns = wsi01_trans.iloc[0]

wsi01_trans = wsi01_trans.drop(index=0)
wsi01_trans = wsi01_trans.reset_index(drop=True)

wsi01_weighted = wsi01_trans[rank_df.index] * rank_df.values

In [69]:
wsi01_trans.loc[:, f"OS_composite_score"] = wsi01_weighted.sum(axis=1)

wsi01_trans = wsi01_trans[["Features", f"OS_composite_score"]]

wsi01_trans = wsi01_trans.rename(columns={"Features": "sampleid"})

wsi01_trans["sampleid"] = [int(x.split("_")[1]) for x in wsi01_trans["sampleid"]]

wsi01_trans = wsi01_trans.sort_values(f"OS_composite_score", ascending=False).reset_index(drop=True)

In [ ]:
wsi01_trans.to_excel("wsi01_pre_OS_composite_scores.xlsx", index=False)